# Notebook 13b: Empirical Socrata API Investigation of Pre-March 2003 Dates
### DAT535 Data-Intensive Systems | Medallion Lakehouse Validation Specification
**Author**: Group 13 | **Kernel**: `dat535` Python Kernel | **Target**: NYC Open Data Socrata APIs

---

## 1. Executive Summary & Objective

In our Silver Layer pipeline (`clean_311.py`), **Boundary Validation & Quarantining (Stage 2)** enforces a hard temporal boundary:
$$\text{created\_at\_utc} \ge \text{2003-03-01T00:00:00Z}$$
Records with timestamps prior to March 1, 2003 are routed to `quarantined_311.parquet` with the diagnostic tag `PRE_311_EPOCH_DATE`.

### Core Questions Investigated:
1. **Do records prior to March 2003 exist in the City of New York's published datasets?**
2. **What does the official NYC 311 Socrata API return when querying candidate dates across historical epochs (`1970`, `1990`, `2000`, `< 2003-03-01`)?**
3. **What is the earliest recorded timestamp in NYC Open Data across live and archived 311 datasets?**
4. **How does the Silver pipeline's quarantine mechanism handle synthetic corrupted dates to preserve lineage?**

In [1]:
import json
import urllib.parse
import urllib.request
from datetime import datetime, timezone
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Live NYC Open Data Socrata API Queries

We query the official NYC Open Data Socrata API across multiple published 311 dataset endpoints:
1. **`erm2-nwe9`**: 311 Service Requests from 2020 to Present (Current Live Primary Endpoint)
2. **`76ig-c548`**: 311 Service Requests from 2010 to 2019 (Historical Decade Archive)
3. **`sqcr-6mww`**: 311 Service Requests for 2004 (The Earliest Annual Archive Published by NYC)

For each endpoint, we test multiple candidate cutoff dates:
* `1970-01-01`: Standard Unix Epoch reset (generated by `0` timestamp values or uninitialized timestamps).
* `1990-01-01`: Pre-Internet legacy era.
* `2000-01-01`: Y2K milestone.
* `2003-03-01`: The official inception of NYC 311 under Mayor Michael Bloomberg.

In [2]:
endpoints = [
    {"id": "erm2-nwe9", "title": "311 Service Requests (2020 to Present)"},
    {"id": "76ig-c548", "title": "311 Service Requests (2010 to 2019)"},
    {"id": "sqcr-6mww", "title": "311 Service Requests (2004 - Earliest Archive)"},
]

test_dates = [
    ("1970-01-01", "Unix Epoch Reset"),
    ("1990-01-01", "Legacy Pre-Internet"),
    ("2000-01-01", "Y2K Era"),
    ("2003-03-01", "NYC 311 Inception (March 2003)"),
]

print("Executing live Socrata API queries across endpoints and dates...")
results = []
for ep in endpoints:
    eid = ep["id"]
    title = ep["title"]
    row = {"Endpoint ID": eid, "Dataset Coverage": title}
    for date_str, desc in test_dates:
        params = {
            "$where": f"created_date < '{date_str}T00:00:00'",
            "$select": "count(*)",
        }
        url = f"https://data.cityofnewyork.us/resource/{eid}.json?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={"User-Agent": "DAT535-Research/1.0"})
        try:
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = json.loads(resp.read().decode())
                cnt = int(data[0].get("count", 0)) if data else 0
                row[f"< {date_str} ({desc})"] = cnt
        except Exception as e:
            row[f"< {date_str} ({desc})"] = f"Error: {e}"
    results.append(row)

df_results = pd.DataFrame(results)
print("\n=== SOCRATA API RECORD COUNT MATRIX FOR DATES PRIOR TO MARCH 2003 ===")
display(df_results)

Executing live Socrata API queries across endpoints and dates...



=== SOCRATA API RECORD COUNT MATRIX FOR DATES PRIOR TO MARCH 2003 ===


,Endpoint ID,Dataset Coverage,< 1970-01-01 (Unix Epoch Reset),< 1990-01-01 (Legacy Pre-Internet),< 2000-01-01 (Y2K Era),< 2003-03-01 (NYC 311 Inception (March 2003))
0,erm2-nwe9,311 Service Requests (2020 to Present),0,0,0,0
1,76ig-c548,311 Service Requests (2010 to 2019),0,0,0,0
2,sqcr-6mww,311 Service Requests (2004 - Earliest Archive),0,0,0,0


## 3. Earliest Timestamp Discovery Across Datasets

To determine the true starting point of municipal service data published by New York City, we query each dataset endpoint sorted ascending by `created_date` (`$order=created_date ASC&$limit=1`).

In [3]:
earliest_records = []
for ep in endpoints:
    eid = ep["id"]
    title = ep["title"]
    params = {
        "$order": "created_date ASC",
        "$limit": 1,
        "$select": "unique_key,created_date,agency,complaint_type,borough,status",
    }
    url = f"https://data.cityofnewyork.us/resource/{eid}.json?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "DAT535-Research/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = json.loads(resp.read().decode())
            if data:
                r = data[0]
                earliest_records.append({
                    "Dataset": title,
                    "Endpoint": eid,
                    "Earliest unique_key": r.get("unique_key"),
                    "Earliest created_date": r.get("created_date"),
                    "Agency": r.get("agency"),
                    "Complaint Type": r.get("complaint_type"),
                    "Status": r.get("status"),
                })
    except Exception as e:
        earliest_records.append({"Dataset": title, "Endpoint": eid, "Error": str(e)})

df_earliest = pd.DataFrame(earliest_records)
print("=== EARLIEST PUBLISHED RECORD IN EACH NYC OPEN DATA DATASET ===")
display(df_earliest)

=== EARLIEST PUBLISHED RECORD IN EACH NYC OPEN DATA DATASET ===


,Dataset,Endpoint,Earliest unique_key,Earliest created_date,Agency,Complaint Type,Status
0,311 Service Requests (2020 to Present),erm2-nwe9,45285347,2020-01-01T00:00:00.000,DOHMH,Food Poisoning,Closed
1,311 Service Requests (2010 to 2019),76ig-c548,15635792,2010-01-01T00:00:00.000,HPD,HEATING,Closed
2,311 Service Requests (2004 - Earliest Archive),sqcr-6mww,4705298,2004-01-01T00:00:00.000,DOB,General Construction/Plumbing,Closed


## 4. Pipeline Quarantine Demonstration: Defensive Boundary Protection

Since 0 records exist prior to March 2003 on NYC Open Data, any record bearing a timestamp before `2003-03-01` that enters a batch is **provably corrupted** (e.g. system clock drift, uninitialized database integer defaults `0` $\rightarrow$ `1970-01-01`, or operator typo).

Below, we demonstrate the exact PySpark boundary validation logic implemented in `clean_311.py`:
Corrupted pre-2003 records are quarantined under `PRE_311_EPOCH_DATE` to protect Silver analytics while preserving full auditability.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, to_timestamp
from src.utils.spark import get_spark_session

spark = get_spark_session("notebook_13b_quarantine_proof")

# Construct mock batch with both valid modern records and corrupted pre-2003 artifacts
mock_data = [
    (50000001, "2024-05-15T14:30:00.000", "2024-05-15T16:30:00.000", "DOT", "Street Condition", "QUEENS"),
    (50000002, "1999-05-12T08:00:00.000", "1999-05-12T09:00:00.000", "DOT", "Street Condition", "BRONX"),       # Corrupted 1999 typo
    (50000003, "1970-01-01T00:00:00.000", "1970-01-01T01:00:00.000", "NYPD", "Noise - Residential", "MANHATTAN"), # Unix Epoch Reset (0 timestamp)
    (50000004, "2024-06-01T10:00:00.000", "2024-06-01T11:00:00.000", "DSNY", "Sanitation Condition", "BROOKLYN"),
]

df_raw = spark.createDataFrame(
    mock_data,
    ["unique_key", "created_date", "closed_date", "agency", "complaint_type", "borough"]
)

launch_epoch = int(datetime(2003, 3, 1, tzinfo=timezone.utc).timestamp())

# Apply Silver Boundary Predicate
df_quarantine_check = df_raw.withColumn("created_ts", to_timestamp(col("created_date")))
cond_pre_epoch = col("created_ts").isNotNull() & (col("created_ts").cast("long") < lit(launch_epoch))

df_classified = df_quarantine_check.withColumn(
    "pipeline_action",
    when(cond_pre_epoch, lit("QUARANTINE (quarantined_311.parquet)")).otherwise(lit("ACCEPT (Silver)"))
).withColumn(
    "quarantine_reason",
    when(cond_pre_epoch, lit("PRE_311_EPOCH_DATE")).otherwise(lit(None))
).select(
    "unique_key", "created_date", "agency", "complaint_type", "pipeline_action", "quarantine_reason"
)

display(df_classified.toPandas())

C:\Users\Azazel\anaconda3\envs\dat535\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


C:\Users\Azazel\anaconda3\envs\dat535\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,unique_key,created_date,agency,complaint_type,pipeline_action,quarantine_reason
0,50000001,2024-05-15T14:30:00.000,DOT,Street Condition,ACCEPT (Silver),NaN
1,50000002,1999-05-12T08:00:00.000,DOT,Street Condition,QUARANTINE (quarantined_311.parquet),PRE_311_EPOCH_DATE
2,50000003,1970-01-01T00:00:00.000,NYPD,Noise - Residential,QUARANTINE (quarantined_311.parquet),PRE_311_EPOCH_DATE
3,50000004,2024-06-01T10:00:00.000,DSNY,Sanitation Condition,ACCEPT (Silver),NaN


## 5. Key Findings & Architectural Conclusions

1. **Empirical Fact**:
   * Direct API queries against all NYC Open Data endpoints (`erm2-nwe9`, `76ig-c548`, and `sqcr-6mww`) return **exactly 0 records** for any date prior to March 1, 2003.
2. **Historical Inception**:
   * The City of New York officially inaugurated the 311 Citizen Service Center on **March 9, 2003** (Mayor Michael Bloomberg administration). The earliest published historical dataset available on NYC Open Data starts in **2004** (`sqcr-6mww`).
3. **Engineering Rationale for Silver Stage 2**:
   * The March 1, 2003 threshold is a **defensive data quality contract**. If an ingestion batch ever receives uninitialized Unix epoch defaults (`1970-01-01`), Excel legacy zeros (`1900-01-01`), or typographically corrupted years (`1999`), the pipeline automatically routes them to `quarantined_311.parquet` under `PRE_311_EPOCH_DATE`.
   * This guarantees that longitudinal time-series analytics, SLA duration calculations, and weather joins in Silver/Gold are never polluted by invalid dates.